# Init

In [0]:
import requests
import json
import time
from requests.exceptions import ConnectionError, HTTPError
import pyspark.sql.functions as F
from pyspark.sql import Row
from datetime import datetime, timezone

SINCE = spark.sql("""
    SELECT min(last_watermark) FROM workspace.bronze.control_table
""").first()[0].strftime("%Y-%m-%dT%H:%M:%SZ")

In [0]:
print(SINCE)

In [0]:
from datetime import datetime, timedelta   # add at the top of the notebook/cell

def extract_data():
    # GitHub token and headers used for all API requests
    token = token = dbutils.secrets.get(scope="github", key="token")
    headers = {
        "Authorization": f"Bearer {token}",
        "Accept": "application/vnd.github.v3+json"
    }

    # All repos to pull data from, keyed by a short name used in filenames/tables
    repos = {
        "vscode": "https://api.github.com/repos/microsoft/vscode",
        "TypeScript": "https://api.github.com/repos/microsoft/TypeScript",
        "PowerToys": "https://api.github.com/repos/microsoft/PowerToys",
        "terminal": "https://api.github.com/repos/microsoft/terminal",
        "PowerShell": "https://api.github.com/repos/PowerShell/PowerShell",
        "runtime": "https://api.github.com/repos/dotnet/runtime",
        "aspnetcore": "https://api.github.com/repos/dotnet/aspnetcore",
    }

    # Data types to pull for each repo
    endpoints = ["repo_data", "languages", "commits", "contributors", "pull_requests", "releases"]

    total_repos = len(repos)
    for repo_index, (repo_name, repo_url) in enumerate(repos.items(), start=1):
        print(f"\n{'#' * 60}")
        print(f"# Repo {repo_index}/{total_repos}: {repo_name}")
        print(f"{'#' * 60}")

        for endpoint in endpoints:
            print(f"\n>>> Starting endpoint '{endpoint}' for repo '{repo_name}'")

            # -----------------------------------------------------------
            # repo_data: single object, one request, no pagination
            # -----------------------------------------------------------
            if endpoint == "repo_data":
                try:
                    with open(f"/Volumes/workspace/bronze/github_json/{repo_name}_repo_data.jsonl", "w") as f:
                        response = requests.get(repo_url, headers=headers)
                        response.raise_for_status()
                        print(f"repo name: {repo_name}, endpoint: {endpoint}, status code: {response.status_code}")
                        repo_data = response.json()
                        f.write(json.dumps(repo_data) + "\n")
                    print(f"<<< Finished endpoint '{endpoint}' for repo '{repo_name}'")
                except requests.exceptions.HTTPError as http_err:
                    print(f'HTTP error occurred: {http_err}')
                except Exception as err:
                    print(f'Other error occurred: {err}')

            # -----------------------------------------------------------
            # languages: single object, one request, no pagination
            # -----------------------------------------------------------
            elif endpoint == "languages":
                try:
                    with open(f"/Volumes/workspace/bronze/github_json/{repo_name}_language.jsonl", "w") as f:
                        response = requests.get(f"{repo_url}/languages", headers=headers)
                        response.raise_for_status()
                        print(f"repo name: {repo_name}, endpoint: {endpoint}, status code: {response.status_code}")
                        language_data = response.json()
                        f.write(json.dumps(language_data) + "\n")
                    print(f"<<< Finished endpoint '{endpoint}' for repo '{repo_name}'")
                except HTTPError as http_err:
                    print(f'HTTP error occurred: {http_err}')
                except Exception as err:
                    print(f'Other error occurred: {err}')

            # -----------------------------------------------------------
            # pull_requests: paginated, no "since" filter (API doesn't
            # support it here) - stops manually on the created_at cutoff
            # -----------------------------------------------------------
            elif endpoint == "pull_requests":
                try:
                    with open(f"/Volumes/workspace/bronze/github_json/{repo_name}_pull_requests.jsonl", "w") as f:
                        page = 1
                        total_records = 0
                        stop = False
                        cutoff_date = SINCE
                        while not stop:
                            response = requests.get(
                                f"{repo_url}/pulls",
                                headers=headers,
                                params={
                                    "state": "all",
                                    "per_page": 100,
                                    "page": page,
                                    "sort": "created",
                                    "direction": "desc"
                                }
                            )
                            response.raise_for_status()
                            page_data = response.json()
                            if len(page_data) == 0:
                                stop = True
                                break
                            for record in page_data:
                                if record["created_at"] < cutoff_date:
                                    stop = True
                                    break
                                f.write(json.dumps(record) + "\n")
                                total_records += 1
                            if page % 20 == 0:
                                print(f"status code {response.status_code}, progress: page {page}, records so far: {total_records}")
                            page += 1
                            time.sleep(1)
                except HTTPError as http_err:
                    print(f'HTTP error occurred: {http_err}')
                except Exception as err:
                    print(f'Other error occurred: {err}')
                print(f"<<< Finished endpoint '{endpoint}' for repo '{repo_name}' - Total records: {total_records}")

            # -----------------------------------------------------------
            # commits / contributors / releases: paginated, uses "since"
            # cutoff where supported by the API.
            # For commits only: after the normal pull, if the oldest commit
            # we got is much newer than the repo creation date, the pull
            # stopped early. In that case we run a second pass (phase 2)
            # that pulls everything older than the oldest commit we have.
            # -----------------------------------------------------------
            else:
                url = f"{repo_url}/{endpoint}"
                total_records = 0
                total_pages = 0
                failed = False
                seen_shas = set()      # dedupe commits by sha (phase 2 overlaps at the boundary)
                oldest_date = None     # oldest commit date pulled so far
                until = None           # set only in phase 2 (backfill)

                with open(f"/Volumes/workspace/bronze/github_json/{repo_name}_{endpoint}.jsonl", "w") as f:
                    for phase in (1, 2):
                        page = 1
                        while True:
                            try:
                                params = {'page': page, 'per_page': 100, 'since': SINCE}
                                if until:
                                    # Backfill window: everything from the beginning up to the oldest commit we have
                                    params['since'] = "2000-01-01T00:00:00Z"
                                    params['until'] = until

                                # Retry on rate limit / transient server errors
                                for attempt in range(5):
                                    response = requests.get(url, headers=headers, params=params, timeout=30)
                                    if response.status_code in (403, 429, 500, 502, 503):
                                        print(f"page {page}: status {response.status_code}, retry {attempt + 1}/5")
                                        time.sleep(30 * (attempt + 1))
                                        continue
                                    break

                                response.raise_for_status()
                                page_data = response.json()
                                if not page_data:
                                    break

                                for record in page_data:
                                    if endpoint == "commits":
                                        # Skip duplicates and track the oldest commit date
                                        if record["sha"] in seen_shas:
                                            continue
                                        seen_shas.add(record["sha"])
                                        d = record["commit"]["committer"]["date"]
                                        if oldest_date is None or d < oldest_date:
                                            oldest_date = d
                                    f.write(json.dumps(record) + "\n")
                                    total_records += 1

                                if page % 100 == 0:
                                    print(f"status code {response.status_code}, progress: page {page}, records so far: {total_records}")
                                page += 1
                                total_pages += 1
                                time.sleep(1)
                            except HTTPError as http_err:
                                print(f'HTTP error occurred: {http_err}')
                                failed = True
                                break
                            except Exception as err:
                                print(f'Other error occurred: {err}')
                                failed = True
                                break

                        # ---- End of phase 1: decide if a backfill is needed ----
                        if phase == 1:
                            # Only commits are backfilled, and only if phase 1 finished without errors
                            if endpoint != "commits" or failed or oldest_date is None or SINCE > "2010-01-01T00:00:00Z":
                                break
                            try:
                                repo_created = requests.get(repo_url, headers=headers, timeout=30).json()["created_at"]
                                created_dt = datetime.fromisoformat(repo_created.replace("Z", "+00:00"))
                                oldest_dt = datetime.fromisoformat(oldest_date.replace("Z", "+00:00"))
                                # Oldest commit more than 30 days after repo creation -> history is missing
                                if oldest_dt > created_dt + timedelta(days=30):
                                    print(f"!!! {repo_name}: oldest commit={oldest_date}, repo created={repo_created} -> backfilling")
                                    until = oldest_date
                                else:
                                    break
                            except Exception as err:
                                print(f"!!! {repo_name}: could not check for missing history: {err}")
                                failed = True
                                break

                if failed:
                    print(f"!!! {repo_name}/{endpoint} FAILED - data is incomplete")
                print(f"<<< Finished endpoint '{endpoint}' for repo '{repo_name}' - Total pages: {total_pages}, Total records: {total_records}")

    print(f"\n{'#' * 60}")
    print("# All repos and endpoints processed")
    print(f"{'#' * 60}")

In [0]:
def build_bronze_table():
    repos_names = ["vscode", "TypeScript", "PowerToys", "terminal", "PowerShell", "runtime", "aspnetcore"]
    endpoints = ['repo_data', 'languages', 'commits', 'contributors', 'pull_requests', 'releases']
    for repo_name in repos_names:
        for endpoint in endpoints:
            if endpoint == "languages":
                with open(f'/Volumes/workspace/bronze/github_json/{repo_name}_language.jsonl') as f:
                    language_dict = json.loads(f.readline())
                rows = [Row(language=lang, bytes=count) for lang, count in language_dict.items()]
                df_languages = spark.createDataFrame(rows)
                df_languages = df_languages.withColumn("ingested_at", F.current_timestamp())
                df_languages.write.mode("overwrite").saveAsTable(f"workspace.bronze.raw_{repo_name}_languages")
                print(f"raw_languages: {df_languages.count()} rows")
            else:
                df = spark.read.json(f'/Volumes/workspace/bronze/github_json/{repo_name}_{endpoint}.jsonl')
                df = df.withColumn("ingested_at", F.current_timestamp())
                df.write.mode("overwrite").saveAsTable(f"workspace.bronze.raw_{repo_name}_{endpoint}")
                print(f"raw_{endpoint}: {df.count()} rows")

In [0]:
# CONTROL TABLE
spark.sql(f"""
          CREATE TABLE IF NOT EXISTS workspace.bronze.control_table(
              pipeline_name STRING,
              last_watermark TIMESTAMP,
              updated_at TIMESTAMP
          )
          USING DELTA
          """)

In [0]:
repos_names = ["vscode", "typescript", "powertoys", "terminal", "powershell", "runtime", "aspnetcore"]
endpoints = ["commits", "pull_requests"]
for repo_name in repos_names:
    for endpoint in endpoints:
        # MERGE INTO CONTROL TABLE
        pipeline_name = f"{repo_name}_{endpoint}"
        spark.sql(f"""
                MERGE INTO workspace.bronze.control_table AS target
                USING(
                    SELECT
                        '{pipeline_name}' AS pipeline_name,
                        CAST('2010-01-01 00:00:00' AS TIMESTAMP) AS last_watermark,
                        current_timestamp() AS updated_at
                ) AS source
                ON target.pipeline_name = source.pipeline_name
                WHEN NOT MATCHED THEN
                INSERT(
                    pipeline_name,
                    last_watermark,
                    updated_at
                ) VALUES (
                    source.pipeline_name,
                    source.last_watermark,
                    source.updated_at
                )
                """)

In [0]:
%sql
SELECT * FROM workspace.bronze.control_table

In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F, Window

CONTROL_TABLE = "workspace.bronze.control_table"


def run_incremental(repo_name, endpoint, key):
    pipeline_name = f"{repo_name}_{endpoint}"
    source_table = f"workspace.bronze.raw_{repo_name}_{endpoint}"
    target_table = f"workspace.bronze.inc_{repo_name}_{endpoint}"

    # STEP 1: Read the last watermark for this pipeline
    watermark_row = (
        spark.table(CONTROL_TABLE)
        .where(F.col("pipeline_name") == pipeline_name)
        .select("last_watermark")
        .first()
    )

    if watermark_row is None or watermark_row["last_watermark"] is None:
        raise RuntimeError(f"No watermark found for {pipeline_name}")

    last_watermark = watermark_row["last_watermark"]

    # STEP 2: Capture a stable upper boundary for this run
    source_df = spark.table(source_table)
    run_upper_bound = source_df.agg(F.max("ingested_at")).first()[0]

    # Nothing new since the last run, so exit early
    if run_upper_bound is None or run_upper_bound <= last_watermark:
        return {"pipeline": pipeline_name, "status": "NO_NEW_DATA"}

    # STEP 3: Read only the new window (last_watermark, run_upper_bound]
    incremental_df = source_df.where(
        (F.col("ingested_at") > F.lit(last_watermark))
        & (F.col("ingested_at") <= F.lit(run_upper_bound))
    )

    # STEP 4: Deduplicate by key, keep the latest row only
    latest_window = Window.partitionBy(key).orderBy(F.col("ingested_at").desc_nulls_last())

    prepared_df = (
        incremental_df
        .withColumn("_row_number", F.row_number().over(latest_window))
        .where(F.col("_row_number") == 1)
        .drop("_row_number")
    )

    # STEP 5: Insert new rows and update existing ones
    if not spark.catalog.tableExists(target_table):
        # First run: target does not exist yet, just create it
        prepared_df.write.saveAsTable(target_table)
    else:
        (
            DeltaTable.forName(spark, target_table).alias("t")
            .merge(prepared_df.alias("s"), f"t.{key} = s.{key}")
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )

    # STEP 6: Advance the watermark ONLY after the merge succeeded
    DeltaTable.forName(spark, CONTROL_TABLE).update(
        condition=F.col("pipeline_name") == pipeline_name,
        set={
            "last_watermark": F.lit(run_upper_bound),
            "updated_at": F.current_timestamp(),
        },
    )

    return {
        "pipeline": pipeline_name,
        "status": "SUCCESS",
        "old_watermark": str(last_watermark),
        "new_watermark": str(run_upper_bound),
    }

In [0]:
def run_all_incremental():
    repos_names = ["vscode", "typescript", "powertoys", "terminal", "powershell", "runtime", "aspnetcore"]
    INCREMENTAL = {"commits": "sha", "pull_requests": "id"}

    results, failed = [], []

    for repo_name in repos_names:
        for endpoint, key in INCREMENTAL.items():
            try:
                results.append(run_incremental(repo_name, endpoint, key))
            except Exception as e:
                failed.append((repo_name, endpoint, str(e)))

    for r in results:
        print(r)

    if failed:
        raise RuntimeError(f"Failed pipelines: {failed}")

In [0]:
extract_data()
build_bronze_table()
run_all_incremental()